<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Способы оплаты</h2>

----

### Вариант задания 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----
Описание задачи: 
Создать базовый класс PaymentMethod в C#, который будет представлять 
различные способы оплаты. На основе этого класса разработать 2-3 производных 
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из 
классов должны быть реализованы новые атрибуты и методы, а также 
переопределены некоторые методы базового класса для демонстрации 
полиморфизма. 
Требования к базовому классу PaymentMethod: 

• Атрибуты: 
1. ID способа оплаты (PaymentMethodId)
2. Название способа оплаты (MethodName)
3. Минимальная сумма (MinAmount). 

• Методы: 
1. ProcessPayment(decimal amount): метод для обработки платежа 
указанной суммы. 
2. CheckMinimumAmount(decimal amount): метод для проверки 
минимальной суммы платежа. 
3. GetPaymentDetails(): метод для получения деталей способа оплаты.
 
Требования к производным классам: 
1. ОнлайнОплата (OnlinePayment): Должен содержать дополнительные 
атрибуты, 
такие 
как 
URL платежной системы (PaymentUrl). 
Метод ProcessPayment() должен быть переопределен для включения URL 
платежной системы в процесс оплаты. 
2. БанковскийПеревод (BankTransfer): Должен содержать дополнительные 
атрибуты, 
такие 
как 
Банковские 
данные 
(BankData). 
Метод CheckMinimumAmount() должен быть переопределен для проверки 
минимальной суммы платежа с учетом банковских комиссий. 
3. Наличные (CashPayment) (если требуется третий класс): Должен содержать 
дополнительные атрибуты, такие как Место выдачи наличных 
(CashPickupPoint). Метод GetPaymentDetails() должен быть переопределен 
для отображения места выдачи наличных.


#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;
using System.Collections.Generic;

public interface IRefundable
{
    bool Refund(decimal amount);
}

public interface ICancelable
{
    void Cancel();
}

public interface ILogger
{
    void Log(string message);
}

public class ConsoleLogger : ILogger
{
    public void Log(string message)
    {
        Console.WriteLine($"[ЛОГ] {message}");
    }
}

public class PaymentMethod
{
    private int _paymentMethodId;
    private string _methodName;
    private decimal _minAmount;
    protected string _currency;
    protected bool _isEnabled;
    protected int _transactionCount;
    protected string _providerName;
    protected DateTime _lastTransactionDate;
    protected bool _requiresKyc;
    protected string _securityLevel;

    public string ProviderName { get { return _providerName; } set { _providerName = value; } }
    public DateTime LastTransactionDate { get { return _lastTransactionDate; } set { _lastTransactionDate = value; } }
    public int PaymentMethodId { get { return _paymentMethodId; } set { _paymentMethodId = value; } }
    public string MethodName { get { return _methodName; } set { _methodName = value; } }
    public decimal MinAmount { get { return _minAmount; } set { if (value >= 0) _minAmount = value; } }
    public string Currency { get { return _currency; } set { _currency = value; } }
    public bool IsEnabled { get { return _isEnabled; } set { _isEnabled = value; } }
    public bool RequiresKyc { get { return _requiresKyc; } set { _requiresKyc = value; } }
    public string SecurityLevel { get { return _securityLevel; } set { _securityLevel = value; } }

    public PaymentMethod(int id, string name, decimal minAmount, string currency, string provider, bool requiresKyc, string securityLevel)
    {
        this.PaymentMethodId = id;
        this.MethodName = name; 
        this.MinAmount = minAmount;
        this.Currency = currency;
        this.ProviderName = provider;
        this.RequiresKyc = requiresKyc;
        this.SecurityLevel = securityLevel;
        this.IsEnabled = true;
        this._transactionCount = 0;
        this._lastTransactionDate = DateTime.MinValue;
    }

    public virtual void ProcessPayment(decimal amount)
    {
        if (!IsEnabled) { Console.WriteLine($"Метод '{MethodName}' отключен."); return; }
        if (CheckMinimumAmount(amount))
        {
            _transactionCount++;
            _lastTransactionDate = DateTime.Now;
            Console.WriteLine($"[УСПЕХ] Платеж через {MethodName} ({ProviderName}) на {amount} {Currency} обработан.");
        }
        else
        {
            Console.WriteLine($"[ОТКАЗ] Платеж на {amount} {Currency} отклонен. Мин. сумма: {MinAmount} {Currency}.");
        }
    }

    public virtual void ProcessPayment(decimal amount, string promoCode)
    {
        Console.WriteLine($"[ИНФО] Получен запрос на оплату {amount} {Currency} с промокодом '{promoCode}'.");
        Console.WriteLine($"[ИНФО] Промокод распознан. Применяется скидка 10%.");
        decimal discountedAmount = amount * 0.9m;
        Console.WriteLine($"[ИНФО] Итоговая сумма к оплате после скидки: {discountedAmount} {Currency}.");
        ProcessPayment(discountedAmount);
    }

    public virtual bool CheckMinimumAmount(decimal amount) { return amount >= MinAmount; }

    public virtual void ShowLastTransactionInfo()
    {
        if (_lastTransactionDate == DateTime.MinValue) Console.WriteLine("Транзакций еще не было.");
        else Console.WriteLine($"Последняя операция по {MethodName} была: {_lastTransactionDate}");
    }

    public int GetTransactionCount() { return _transactionCount; }
    public void Disable() { IsEnabled = false; }

    public virtual void VerifyKyc() { Console.WriteLine($"Проверка KYC для {MethodName}: {(RequiresKyc ? "Требуется" : "Не требуется")}"); }
    public virtual string GetSecurityInfo() { return $"Уровень безопасности: {SecurityLevel}"; }
}

public class OnlinePayment : PaymentMethod, IRefundable, ICancelable
{
    private string _paymentUrl;
    private decimal _transactionFee;
    private int _maxDailyTransactions;
    private bool _is3DSecureEnabled;
    private int _sessionTimeout;
    private string _lastIp;

    public string PaymentUrl { get { return _paymentUrl; } set { _paymentUrl = value; } }
    public int MaxDailyTransactions { get { return _maxDailyTransactions; } set { _maxDailyTransactions = value; } }
    public bool Is3DSecureEnabled { get { return _is3DSecureEnabled; } set { _is3DSecureEnabled = value; } }
    public int SessionTimeout { get { return _sessionTimeout; } set { _sessionTimeout = value; } }
    public string LastIp { get { return _lastIp; } set { _lastIp = value; } }

    public OnlinePayment(int id, string name, decimal minAmount, string currency, string provider, bool requiresKyc, string securityLevel, string url, decimal fee, int maxTrans, int timeout, string ip) 
        : base(id, name, minAmount, currency, provider, requiresKyc, securityLevel)
    {
        this.PaymentUrl = url;
        this._transactionFee = fee;
        this.MaxDailyTransactions = maxTrans;
        this._is3DSecureEnabled = true;
        this.SessionTimeout = timeout;
        this.LastIp = ip;
    }

    public override void ProcessPayment(decimal amount)
    {
        if (GetTransactionCount() >= MaxDailyTransactions) { Console.WriteLine($"[ОТКАЗ] Превышен дневной лимит операций ({MaxDailyTransactions})."); return; }
        if (!IsEnabled) { Console.WriteLine($"[ОТКАЗ] Метод '{MethodName}' отключен."); return; }
        base.ProcessPayment(amount);
        Console.WriteLine($"--> Деньги списаны по URL: {PaymentUrl} (Комиссия сервиса: {_transactionFee} {Currency})");
    }

    public override void ProcessPayment(decimal amount, string promoCode)
    {
        Console.WriteLine($"[ОНЛАЙН] Проверка промокода {promoCode} через API... Успешно!");
        base.ProcessPayment(amount, promoCode);
    }

    public void Toggle3DSecure() { _is3DSecureEnabled = !_is3DSecureEnabled; Console.WriteLine($"3D Secure для {MethodName}: {(_is3DSecureEnabled ? "ВКЛ" : "ВЫКЛ")}"); }
    public bool Refund(decimal amount) { Console.WriteLine($"[ВОЗВРАТ] {amount} {Currency} возвращены на карту."); return true; }
    
    public void ClearSession() { Console.WriteLine($"Сессия {MethodName} очищена. Таймаут: {SessionTimeout} сек."); }
    public void CheckFraud() { Console.WriteLine($"Проверка IP {LastIp} на фрод: OK"); }

    void ICancelable.Cancel()
    {
        Console.WriteLine($"[ОТМЕНА] Онлайн-платеж {MethodName} отменен на стороне платежного шлюза.");
    }
}

public class BankTransfer : PaymentMethod, ICancelable
{
    private decimal _bankData;
    private string _swiftCode;
    private string _correspondentAccount;
    private int _processingTimeHours;
    private string _branchCode;
    private bool _isInternational;

    public decimal BankData { get { return _bankData; } set { _bankData = value; } }
    public string SwiftCode { get { return _swiftCode; } set { _swiftCode = value; } }
    public string CorrespondentAccount { get { return _correspondentAccount; } set { _correspondentAccount = value; } }
    public string BranchCode { get { return _branchCode; } set { _branchCode = value; } }
    public bool IsInternational { get { return _isInternational; } set { _isInternational = value; } }

    public BankTransfer(int id, string name, decimal minAmount, string currency, string provider, bool requiresKyc, string securityLevel, decimal bankData, string swift, string corrAcc, int time, string branch, bool isInt) 
        : base(id, name, minAmount, currency, provider, requiresKyc, securityLevel)
    {
        this.BankData = bankData;
        this._swiftCode = swift;
        this._correspondentAccount = corrAcc;
        this._processingTimeHours = time;
        this.BranchCode = branch;
        this.IsInternational = isInt;
    }

    public override bool CheckMinimumAmount(decimal amount)
    {
        decimal commission = amount / 100 * BankData;
        decimal amountAfterCommission = amount - commission;
        if (amountAfterCommission >= MinAmount) return true;
        Console.WriteLine($"[ОТКАЗ] Недостаточно средств. Комиссия: {commission}. Остаток: {amountAfterCommission} (нужно {MinAmount})");
        return false;
    }

    public override void ShowLastTransactionInfo()
    {
        base.ShowLastTransactionInfo();
        Console.WriteLine($"(Детали банка: SWIFT {_swiftCode}, Корр.счет {_correspondentAccount})");
    }

    public void ValidateBankDetails() { if (_swiftCode.Length >= 8) Console.WriteLine($"[ПРОВЕРКА] SWIFT-код {_swiftCode} валиден."); else Console.WriteLine("[ОШИБКА] Неверный формат SWIFT."); }
    public void EstimateArrivalTime() { Console.WriteLine($"Ожидаемое время зачисления: {_processingTimeHours} ч."); }
    public void PrintSwiftDetails() { Console.WriteLine($"SWIFT детали: {_swiftCode}, Филиал: {BranchCode}, Международный: {IsInternational}"); }
    public void CheckSanctions() { Console.WriteLine("Проверка по санкционным спискам: пройдена."); }

    void ICancelable.Cancel()
    {
        Console.WriteLine($"[ОТМЕНА] Банковский перевод {MethodName} отменен, средства возвращены на счет отправителя.");
    }
}

public class CashPayment : PaymentMethod, ICancelable
{
    private string _cashPickupPoint;
    private string _operatorName;
    private string _cashierName;
    private decimal _cashInDrawer;
    private string _receiptPrinterModel;
    private decimal _dailyLimit;

    public string CashPickupPoint { get { return _cashPickupPoint; } set { _cashPickupPoint = value; } }
    public string CashierName { get { return _cashierName; } set { _cashierName = value; } }
    public decimal CashInDrawer { get { return _cashInDrawer; } set { _cashInDrawer = value; } }
    public string ReceiptPrinterModel { get { return _receiptPrinterModel; } set { _receiptPrinterModel = value; } }
    public decimal DailyLimit { get { return _dailyLimit; } set { _dailyLimit = value; } }

    public CashPayment(int id, string name, decimal minAmount, string currency, string provider, bool requiresKyc, string securityLevel, string pickupPoint, string opName, string cashier, decimal cash, string printer, decimal limit) 
        : base(id, name, minAmount, currency, provider, requiresKyc, securityLevel)
    {
        this._cashPickupPoint = pickupPoint;
        this._operatorName = opName;
        this._cashierName = cashier;
        this._cashInDrawer = cash;
        this.ReceiptPrinterModel = printer;
        this.DailyLimit = limit;
    }

    public override void ProcessPayment(decimal amount)
    {
        base.ProcessPayment(amount); 
        _cashInDrawer += amount;
        Console.WriteLine($"--> Деньги приняты. Теперь у кассира {_cashierName}: {_cashInDrawer} {Currency}");
    }

    public void RequestCashier() { Console.WriteLine($"[ВЫЗОВ] Кассир {_cashierName} приглашается к окну."); }
    public void PrintReceipt() { Console.WriteLine($"Чек распечатан на {ReceiptPrinterModel} оператором {_operatorName}"); }
    public void EmptyDrawer() { Console.WriteLine($"Инкассация: {_cashInDrawer} {Currency} изъято из кассы."); _cashInDrawer = 0; }
    public void CalibratePrinter() { Console.WriteLine($"Калибровка принтера {ReceiptPrinterModel} завершена."); }

    void ICancelable.Cancel()
    {
        Console.WriteLine($"[ОТМЕНА] Наличная операция {MethodName} отменена, чек аннулирован.");
    }
}

public class CryptoPayment : OnlinePayment, ICancelable
{
    private double _networkFee;
    private int _confirmationsRequired;
    private string _explorerUrl;
    private bool _isColdStorage;

    public double NetworkFee { get { return _networkFee; } set { _networkFee = value; } }
    public int ConfirmationsRequired { get { return _confirmationsRequired; } set { _confirmationsRequired = value; } }
    public string ExplorerUrl { get { return _explorerUrl; } set { _explorerUrl = value; } }
    public bool IsColdStorage { get { return _isColdStorage; } set { _isColdStorage = value; } }

    public CryptoPayment(int id, string name, decimal minAmount, string currency, string provider, bool requiresKyc, string securityLevel, string url, decimal fee, int maxTrans, int timeout, string ip, double netFee, int conf, string explorer, bool cold) 
        : base(id, name, minAmount, currency, provider, requiresKyc, securityLevel, url, fee, maxTrans, timeout, ip)
    {
        this._networkFee = netFee;
        this._confirmationsRequired = conf;
        this.ExplorerUrl = explorer;
        this.IsColdStorage = cold;
    }

    public override void ProcessPayment(decimal amount)
    {
        Console.WriteLine($"[КРИПТО] Попытка оплаты {amount} {Currency}");
        Console.WriteLine($"[КРИПТО] Добавляем комиссию сети: {_networkFee} {Currency}");
        decimal totalAmount = amount + (decimal)_networkFee;
        Console.WriteLine($"[КРИПТО] Итоговая сумма к списанию: {totalAmount} {Currency}");
        Console.WriteLine($"[КРИПТО] Ожидание {_confirmationsRequired} подтверждений сети...");
        base.ProcessPayment(totalAmount); 
    }

    public void CheckNetworkCongestion() { Console.WriteLine($"[СТАТУС СЕТИ] Загруженность средняя. Комиссия сети: {_networkFee} {Currency}."); }
    public void VerifyBlockchain() { Console.WriteLine($"Проверка транзакции в блокчейне ({ExplorerUrl})... Подтверждено."); }
    public void GenerateNewAddress() { Console.WriteLine($"Сгенерирован новый адрес кошелька. Холодное хранение: {IsColdStorage}"); }
    public void CheckMempool() { Console.WriteLine("Проверка мемпула: транзакция не зависла."); }

    void ICancelable.Cancel()
    {
        Console.WriteLine($"[ОТМЕНА] Крипто-транзакция {MethodName} отменена (если еще не подтверждена в мемпуле).");
    }
}

public class PaymentTerminal<T> where T : PaymentMethod
{
    private List<T> _availableMethods = new List<T>();
    private readonly ILogger _logger;

    public PaymentTerminal(ILogger logger)
    {
        _logger = logger;
    }

    public void AddMethod(T method) 
    { 
        _availableMethods.Add(method); 
        _logger.Log($"Метод {method.MethodName} добавлен в терминал.");
    }

    public void ProcessAllAvailable(decimal amount)
    {
        _logger.Log($"Запуск обработки платежа на сумму {amount} через все доступные методы.");
        foreach (var method in _availableMethods) { method.ProcessPayment(amount); }
    }
}

public class Customer
{
    public string Name { get; set; }
    private decimal _balance;
    private string _loyaltyTier;
    private readonly ILogger _logger;

    public decimal Balance { get { return _balance; } set { _balance = value; } }
    public string LoyaltyTier { get { return _loyaltyTier; } set { _loyaltyTier = value; } }

    public Customer(string name, decimal balance, string tier, ILogger logger)
    {
        Name = name;
        _balance = balance;
        _loyaltyTier = tier;
        _logger = logger;
    }

    public void MakePurchase(PaymentMethod method, decimal amount)
    {
        _logger.Log($"Клиент {Name} инициирует платеж на {amount} {method.Currency}.");
        method.ProcessPayment(amount);
    }

    public void RequestRefund(IRefundable refundableMethod, decimal amount)
    {
        _logger.Log($"Клиент {Name} запрашивает возврат {amount}.");
        refundableMethod.Refund(amount);
    }

    public void AddFunds(decimal amount)
    {
        _balance += amount;
        _logger.Log($"Баланс {Name} пополнен на {amount}. Текущий баланс: {_balance} руб.");
    }
}

ILogger logger = new ConsoleLogger();

BankTransfer bank = new BankTransfer(2, "Банковский перевод", 50, "RUB", "Сбербанк", true, "Высокий", 5, "SABRRUMM", "40702810...", 24, "7701", false);
CashPayment cash = new CashPayment(3, "Наличные", 100, "RUB", "Газпромбанк", false, "Средний", "Касса №1", "Ольга", "Лариса", 5000, "Epson TM-T20", 50000);
OnlinePayment online = new OnlinePayment(1, "Онлайн оплата", 10, "RUB", "ЮKassa", true, "Высокий", "https://pay.ru", 2, 10, 300, "192.168.1.1");
CryptoPayment crypto = new CryptoPayment(4, "Крипто-оплата", 500, "USDT", "Binance", false, "Средний", "https://crypto.gate", 1, 5, 600, "10.0.0.5", 0.5, 3, "https://etherscan.io", true);

Customer buyer = new Customer("Кирилл", 10000, "Gold", logger);
PaymentTerminal<PaymentMethod> terminal = new PaymentTerminal<PaymentMethod>(logger);

Console.WriteLine("=== Демонстрация новых методов классов ===");
bank.PrintSwiftDetails();
bank.CheckSanctions();
cash.CalibratePrinter();
crypto.GenerateNewAddress();
crypto.CheckMempool();

Console.WriteLine("\n=== Демонстрация явной реализации интерфейса ICancelable ===");
((ICancelable)online).Cancel();
((ICancelable)bank).Cancel();
((ICancelable)cash).Cancel();
((ICancelable)crypto).Cancel();

Console.WriteLine("\n=== Демонстрация управления зависимостями ===");
terminal.AddMethod(bank);
terminal.AddMethod(cash);
terminal.AddMethod(crypto);
terminal.AddMethod(online);

buyer.MakePurchase(online, 1000);
terminal.ProcessAllAvailable(500);

=== Демонстрация новых методов классов ===
SWIFT детали: SABRRUMM, Филиал: 7701, Международный: False
Проверка по санкционным спискам: пройдена.
Калибровка принтера Epson TM-T20 завершена.
Сгенерирован новый адрес кошелька. Холодное хранение: True
Проверка мемпула: транзакция не зависла.

=== Демонстрация явной реализации интерфейса ICancelable ===
[ОТМЕНА] Онлайн-платеж Онлайн оплата отменен на стороне платежного шлюза.
[ОТМЕНА] Банковский перевод Банковский перевод отменен, средства возвращены на счет отправителя.
[ОТМЕНА] Наличная операция Наличные отменена, чек аннулирован.
[ОТМЕНА] Крипто-транзакция Крипто-оплата отменена (если еще не подтверждена в мемпуле).

=== Демонстрация управления зависимостями ===
[ЛОГ] Метод Банковский перевод добавлен в терминал.
[ЛОГ] Метод Наличные добавлен в терминал.
[ЛОГ] Метод Крипто-оплата добавлен в терминал.
[ЛОГ] Метод Онлайн оплата добавлен в терминал.
[ЛОГ] Клиент Кирилл инициирует платеж на 1000 RUB.
[УСПЕХ] Платеж через Онлайн оплата (ЮKass